# 🔬 DeepTrace — Swin Transformer Fine-Tuning on CIFAKE Dataset
This notebook fine-tunes **`microsoft/swin-tiny-patch4-window7-224`** on the **CIFAKE dataset (120,000 images)** for state-of-the-art Deepfake & AI-generated image detection.

### ⚡ Prerequisites:
Make sure your Colab Runtime is set to **T4 GPU** (`Runtime > Change runtime type > T4 GPU`).

In [ ]:
# Step 1: Install Required Dependencies
!pip install -q transformers datasets accelerate torchvision evaluate kagglehub scikit-learn huggingface_hub

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# Step 1.5: Authenticate with HuggingFace for full-speed downloads
# Paste your token when prompted (get one at: https://huggingface.co/settings/tokens)
from huggingface_hub import login
login()

In [ ]:
# Step 2: Download the CIFAKE Dataset (120,000 Real vs. AI Images)
import os
import kagglehub
from datasets import load_dataset

print("Downloading CIFAKE dataset via kagglehub...")
try:
    cifake_path = kagglehub.dataset_download("birdy654/cifake-real-and-ai-generated-synthetic-images")
    print(f"Dataset downloaded locally to: {cifake_path}")
    dataset = load_dataset("imagefolder", data_dir=cifake_path)
except Exception as e:
    print(f"kagglehub failed ({e}), loading from HuggingFace mirror...")
    dataset = load_dataset("jlbaker361/cifake")

print(dataset)

# Verify actual label mapping from the loaded dataset
# imagefolder assigns labels alphabetically: FAKE=0, REAL=1
# HuggingFace mirror uses: REAL=0, FAKE=1
if hasattr(dataset["train"].features["label"], "names"):
    class_names = dataset["train"].features["label"].names
    print(f"Dataset class names (by index): {list(enumerate(class_names))}")
else:
    class_names = None
    print("No ClassLabel metadata found — will use default mapping.")

In [ ]:
# Step 3: Setup Swin Image Processor & Data Transforms
from transformers import AutoImageProcessor
from torchvision.transforms import (
    Compose, Resize, CenterCrop, RandomResizedCrop, 
    RandomHorizontalFlip, ToTensor, Normalize
)

MODEL_ID = "microsoft/swin-tiny-patch4-window7-224"  # Can also use "microsoft/swin-base-patch4-window7-224"
processor = AutoImageProcessor.from_pretrained(MODEL_ID)

size = (
    processor.size["height"], processor.size["width"]
) if "height" in processor.size else (
    processor.size["shortest_edge"], processor.size["shortest_edge"]
)

normalize = Normalize(mean=processor.image_mean, std=processor.image_std)

train_transforms = Compose([
    RandomResizedCrop(size),
    RandomHorizontalFlip(),
    ToTensor(),
    normalize,
])

val_transforms = Compose([
    Resize(size),
    CenterCrop(size),
    ToTensor(),
    normalize,
])

def preprocess_train(batch):
    batch["pixel_values"] = [train_transforms(img.convert("RGB")) for img in batch["image"]]
    return batch

def preprocess_val(batch):
    batch["pixel_values"] = [val_transforms(img.convert("RGB")) for img in batch["image"]]
    return batch

# Apply transforms lazily (saves RAM)
train_ds = dataset["train"].with_transform(preprocess_train)
val_ds = dataset["test"].with_transform(preprocess_val)
print("Transforms initialized successfully!")

In [ ]:
# Step 4: Initialize Swin Model for Binary Classification (Real vs. Fake)
from transformers import AutoModelForImageClassification

# Dynamically build label mapping from the dataset's actual class order
# This ensures correctness regardless of whether kagglehub or HuggingFace loaded
if class_names is not None:
    # Normalize names: 'FAKE' -> 'Fake', 'REAL' -> 'Real'
    id2label = {i: name.capitalize() for i, name in enumerate(class_names)}
    label2id = {v: k for k, v in id2label.items()}
else:
    # Safe fallback matching the HuggingFace mirror convention
    id2label = {0: "Real", 1: "Fake"}
    label2id = {"Real": 0, "Fake": 1}

print(f"Using label mapping: id2label={id2label}, label2id={label2id}")

model = AutoModelForImageClassification.from_pretrained(
    MODEL_ID,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)
print("Model loaded with binary classification head!")

In [ ]:
# Step 5: Setup Evaluation Metrics (Accuracy, Precision, Recall, F1)
import numpy as np
import evaluate

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    f1 = f1_metric.compute(predictions=preds, references=labels, average="weighted")["f1"]
    prec = precision_metric.compute(predictions=preds, references=labels, average="weighted")["precision"]
    rec = recall_metric.compute(predictions=preds, references=labels, average="weighted")["recall"]
    return {"accuracy": acc, "f1": f1, "precision": prec, "recall": rec}

def collate_fn(examples):
    pixel_values = torch.stack([example["pixel_values"] for example in examples])
    labels = torch.tensor([example["label"] for example in examples])
    return {"pixel_values": pixel_values, "labels": labels}

In [ ]:
# Step 6: Configure Training Arguments & Trainer
from transformers import TrainingArguments, Trainer

OUTPUT_DIR = "./swin-deeptrace-finetuned"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=32,
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    warmup_ratio=0.1,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    save_total_limit=1,
    fp16=torch.cuda.is_available(),  # Accelerated mixed-precision training on GPU
    push_to_hub=False,
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=processor,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
)
print("Trainer is ready!")

In [ ]:
# Step 7: Train the Model
print("Starting Swin Transformer fine-tuning...")
trainer.train()

In [ ]:
# Step 8: Evaluate on Test Set
metrics = trainer.evaluate()
print("\n--- FINAL EVALUATION METRICS ---")
for k, v in metrics.items():
    print(f"{k}: {v}")

In [ ]:
# Step 9: Save Model and Create 1-Click Download Zip
import shutil
from google.colab import files

print(f"Saving fine-tuned model and processor to {OUTPUT_DIR}...")
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)

zip_filename = "swin_deeptrace_model.zip"
print(f"Creating zip archive '{zip_filename}'...")
shutil.make_archive("swin_deeptrace_model", "zip", OUTPUT_DIR)

# Trigger automatic download in browser
files.download(zip_filename)
print("\nDone! Extract swin_deeptrace_model.zip into your project's `backend/models/swin-finetuned` directory.")